In [0]:
catalog = 'car_workshop'
dim_schema = 'dim'
fact_schema = 'fact'

In [0]:
DIM_TABLES = [
    'dim_locations',
    'dim_employees',
    'dim_customers',
    'dim_vehicles',
    'dim_products',
    'dim_services',
    'dim_suppliers',
]


In [0]:
FACT_TABLES = [
    'fact_work_orders',           # partitioned year/month
    'fact_work_order_items',
    'fact_sales_transactions',    # partitioned year/month
    'fact_sales_items',
    'fact_invoices',              # partitioned year/month
    'fact_payments',              # partitioned year/month
    'fact_inventory_movements',   # partitioned year/month
    'fact_appointments',          # partitioned year/month
    'fact_purchase_orders',       # partitioned year
    'fact_purchase_order_items',
    'fact_customer_feedback',
    'fact_loyalty_program',
    'fact_employee_schedules',
]


In [0]:
%sql
with cte as(
select distinct *,
row_number() over (partition by customer_code order by registration_date desc) as rn from car_workshop.dim.dim_customers)
delete from car_workshop.dim.dim_customers
where customer_code in (select customer_code from cte where rn > 1)

-- where customer_code is null

In [0]:
%sql

delete from car_workshop.dim.dim_locations
where location_code is null

In [0]:
data = []
for i in DIM_TABLES:
    count = spark.sql(f"select count(*) as cnt from {catalog}.{dim_schema}.{i}").collect()[0]['cnt']
    distinct = spark.sql(f"select count(distinct *) as dst from {catalog}.{dim_schema}.{i}").collect()[0]['dst']
    data.append((i, count, distinct))

df = spark.createDataFrame(data, ["table_name", "count", "distinct_count"])
display(df)


In [0]:
data = []
for i in FACT_TABLES:
    count = spark.sql(f"select count(*) as cnt from {catalog}.{fact_schema}.{i}").collect()[0]['cnt']
    data.append((i, count))

df = spark.createDataFrame(data, ["table_name", "count"])
display(df)
